In [2]:
import pandas as pd
import numpy as np
import pickle
import os

# Load data and model
df = pd.read_csv('data/processed/driver_skill_features.csv')

# Use the new 2026 Regression Model
model_path = 'data/processed/model_regression_2026.pkl'
with open(model_path, 'rb') as f:
    model = pickle.load(f)

features = [
    'GridPosition', 'QualiGapToPole', 'SeasonPointsSoFar', 
    'TeamSeasonPoints', 'AvgFinish_Last3', 'AvgFinish_Last5', 
    'DNFRate_Last5', 'SkillScore', 'ConstructorGap', 'DriverPointsRank'
]

print(f"Loaded Regression Model: {model_path} ✓")
print(f"Data available for years: {df['Year'].unique()}")


Loaded ✓


In [1]:
import os
print(os.getcwd())

os.chdir(r'C:\Users\91917\OneDrive\Desktop\f1-predictor')
print(os.getcwd())

c:\Users\91917\OneDrive\Desktop\f1-predictor\notebooks
C:\Users\91917\OneDrive\Desktop\f1-predictor


In [10]:
def what_if_swap(driver1, driver2, race, year):
    race_data = df[(df['Race'] == race) & (df['Year'] == year)].copy()
    
    if race_data.empty:
        print(f"Race not found: {race} {year}")
        return
    
    d1_mask = race_data['Abbreviation'] == driver1
    d2_mask = race_data['Abbreviation'] == driver2
    
    if not any(d1_mask) or not any(d2_mask):
        print(f"Drivers {driver1} or {driver2} not found in this race.")
        return
        
    d1_team = race_data[d1_mask]['TeamName'].values[0] if 'TeamName' in race_data.columns else 'Team1'
    d2_team = race_data[d2_mask]['TeamName'].values[0] if 'TeamName' in race_data.columns else 'Team2'
    
    print(f"\nSwapping {driver1} ↔ {driver2}")
    print(f"Race: {race} {year}\n")
    
    # Original predictions (Numerical Position)
    orig_pos = model.predict(race_data[features])
    
    # Swap car stats
    swapped = race_data.copy()
    # These features represent the 'car' / 'team context'
    cols_to_swap = [
        'GridPosition', 'QualiGapToPole', 'TeamSeasonPoints',
        'ConstructorGap', 'TeamMedianPosition'
    ]
    
    for col in cols_to_swap:
        if col in swapped.columns:
            d1_val = race_data.loc[d1_mask, col].values[0]
            d2_val = race_data.loc[d2_mask, col].values[0]
            swapped.loc[d1_mask, col] = d2_val
            swapped.loc[d2_mask, col] = d1_val
    
    swap_pos = model.predict(swapped[features])
    
    comparison = pd.DataFrame({
        'Driver': race_data['Abbreviation'].values,
        'ActualPos': race_data['Position'].values.astype(int),
        'PredictedPos_Orig': orig_pos.round(1),
        'PredictedPos_Swap': swap_pos.round(1),
        'Delta': (swap_pos - orig_pos).round(1)
    })
    
    # Sort by PredictedPos_Swap (The new finishing order)
    comparison = comparison.sort_values('PredictedPos_Swap').reset_index(drop=True)
    comparison.index += 1
    comparison.index.name = 'FinalRank'
    
    print("=== PREDICTED GRID AFTER SWAP ===")
    print(comparison.to_string())

what_if_swap('VER', 'NOR', 'Bahrain Grand Prix', 2024)



Swapping NOR (McLaren) ↔ VER (Red Bull Racing)
Race: Bahrain Grand Prix 2023

=== FULL PREDICTED GRID AFTER SWAP ===
             Driver                       Team  ActualPosition  OriginalProb%  SwappedProb%  Change%
PredictedPos                                                                                        
1               PER            Red Bull Racing               2           86.8          86.8      0.0
2               LEC                    Ferrari              19           85.0          85.0      0.0
3               NOR  Red Bull Racing (swapped)              17            3.2          83.8     80.5
4               SAI                    Ferrari               4           56.8          56.8      0.0
5               RUS                   Mercedes               7           40.6          40.6      0.0
6               HAM                   Mercedes               5           29.2          29.2      0.0
7               STR               Aston Martin               6           2

In [11]:
def custom_scenario(driver, new_team_driver, new_grid, race, year):
    """
    Place a driver in another car at a custom grid position
    Example: What if NOR started P1 in VER's Red Bull?
    custom_scenario('NOR', 'VER', 1, 'Bahrain Grand Prix', 2023)
    """
    race_data = df[(df['Race'] == race) & (df['Year'] == year)].copy()
    
    if race_data.empty:
        print(f"Race not found: {race} {year}")
        return
    
    # Get the car we're putting the driver into
    car_mask = race_data['Abbreviation'] == new_team_driver
    driver_mask = race_data['Abbreviation'] == driver
    
    car_team = race_data[car_mask]['TeamName'].values[0]
    
    print(f"\nScenario: {driver} starts P{new_grid} in the {car_team}")
    print(f"Race: {race} {year}\n")
    
    # Original predictions
    orig_probs = model.predict_proba(race_data[features])[:, 1]
    
    # Build scenario
    scenario = race_data.copy()
    
    # Give driver the car stats of new_team_driver
    car_cols = [
        'TeamSeasonPoints',
        'ConstructorGap',
        'TeamMedianPosition'
    ]
    
    for col in car_cols:
        if col in scenario.columns:
            car_val = race_data.loc[car_mask, col].values[0]
            scenario.loc[driver_mask, col] = car_val
    
    # Set custom grid position and quali gap
    pole_gap = race_data.loc[car_mask, 'QualiGapToPole'].values[0]
    scenario.loc[driver_mask, 'GridPosition'] = new_grid
    scenario.loc[driver_mask, 'QualiGapToPole'] = pole_gap
    
    scenario_probs = model.predict_proba(scenario[features])[:, 1]
    
    # Display
    display_team = race_data['TeamName'].copy().values.tolist()
    for i, abbr in enumerate(race_data['Abbreviation'].values):
        if abbr == driver:
            display_team[i] = f"{car_team} (custom)"
    
    comparison = pd.DataFrame({
        'Driver': race_data['Abbreviation'].values,
        'Team': display_team,
        'ActualPosition': race_data['Position'].values.astype(int),
        'OriginalProb%': (orig_probs * 100).round(1),
        'ScenarioProb%': (scenario_probs * 100).round(1),
        'Change%': ((scenario_probs - orig_probs) * 100).round(1)
    }).sort_values('ScenarioProb%', ascending=False).reset_index(drop=True)
    
    comparison.index += 1
    comparison.index.name = 'PredictedPos'
    
    print("=== FULL PREDICTED GRID ===")
    print(comparison.to_string())
    
    print("\n=== SCENARIO IMPACT ===")
    changed = comparison[comparison['Change%'] != 0.0]
    print(changed.to_string())

In [14]:
# NOR starts P1 in the Red Bull
#custom_scenario('NOR', 'VER', 4, 'Bahrain Grand Prix', 2023)

# HAM starts P1 in the Red Bull
custom_scenario('HAM', 'VER', 1, 'Bahrain Grand Prix', 2023)

# VER starts P20 in the McLaren
custom_scenario('VER', 'NOR', 20, 'Bahrain Grand Prix', 2023)


Scenario: HAM starts P1 in the Red Bull Racing
Race: Bahrain Grand Prix 2023

=== FULL PREDICTED GRID ===
             Driver                      Team  ActualPosition  OriginalProb%  ScenarioProb%  Change%
PredictedPos                                                                                        
1               HAM  Red Bull Racing (custom)               5           29.2           92.5     63.3
2               PER           Red Bull Racing               2           86.8           86.8      0.0
3               LEC                   Ferrari              19           85.0           85.0      0.0
4               VER           Red Bull Racing               1           77.7           77.7      0.0
5               SAI                   Ferrari               4           56.8           56.8      0.0
6               RUS                  Mercedes               7           40.6           40.6      0.0
7               STR              Aston Martin               6           24.6         